In [ ]:
from datascience import *
import numpy as np

import matplotlib.pyplot as plots
%matplotlib inline
plots.style.use('fivethirtyeight')

import plotly.express as px
import plotly.graph_objects as go

## Functions

In [ ]:
def standard_units(x):
    """Converts an array x to standard units"""
    return (x - np.mean(x)) / np.std(x)

def correlation(t, x, y):
    x_su = standard_units(t.column(x))
    y_su = standard_units(t.column(y))
    return np.mean(x_su * y_su)

def slope(t, x, y):
    """t is a table; x and y are column labels"""
    r = correlation(t, x, y)
    slope_xy = r * np.std(t.column(y)) / np.std(t.column(x))
    return slope_xy

def intercept(t, x, y):
    """t is a table; x and y are column labels"""
    slope_xy = slope(t, x, y)
    intercept_xy = np.average(t.column(y) - slope_xy * np.average(t.column(x)))
    return intercept_xy

## Demographics: One Predictor

In [ ]:
demographics = Table.read_table('/home/jovyan/shared/Data/district_demographics2016.csv')
demographics.show(5)

In [ ]:
demographics.scatter('College%', 'Median Income', fit_line=True)
correlation(demographics, 'College%', 'Median Income')

In [ ]:
demographics.scatter('Percent voting for Clinton', 'Median Income', fit_line=True)
correlation(demographics, 'Percent voting for Clinton', 'Median Income')

In [ ]:
Clinton_slope = slope(demographics, 'Percent voting for Clinton', 'Median Income')
Clinton_intercept = intercept(demographics, 'Percent voting for Clinton', 'Median Income')

Clinton_slope, Clinton_intercept

In [ ]:
def demographics_Clinton_rmse(any_slope, any_intercept):
    x = demographics.column('Percent voting for Clinton')
    y = demographics.column('Median Income')
    estimate = any_slope*x + any_intercept
    return (np.mean((y - estimate) ** 2)) ** 0.5

In [ ]:
minimize(demographics_Clinton_rmse)

In [ ]:
best_slope = slope(demographics, 'Percent voting for Clinton', 'Median Income')
best_intercept = intercept(demographics, 'Percent voting for Clinton', 'Median Income')

slopes = np.arange(best_slope - 1000, best_slope + 1000, 20)

intercepts = np.arange(best_intercept - 20000, best_intercept + 20000, 400)

rmse_values = make_array()

for any_intercept in intercepts:
    row = make_array()

    for any_slope in slopes:
        row = np.append(row, demographics_Clinton_rmse(any_slope, any_intercept))

    rmse_values = np.append(rmse_values, row)

rmse_values = rmse_values.reshape(intercepts.size, slopes.size)

minimum_rmse = demographics_Clinton_rmse(best_slope, best_intercept)

fig = go.Figure()

fig.add_trace(
    go.Surface(x = slopes, y = intercepts, z = rmse_values))

fig.add_trace(
    go.Scatter3d(
        x = make_array(best_slope),
        y = make_array(best_intercept),
        z = make_array(minimum_rmse)))


fig.update_layout(
    title='Minimizing the Standard Deviation of Errors',
    scene=dict(
        xaxis_title='Slope',
        yaxis_title='Intercept',
        zaxis_title='RMSE'
    ),
    width=1000,
    height=750
)

## Demographics: Two Predictors

In [ ]:
px.scatter_3d(
    demographics.to_df(), 
    x = 'College%', 
    y = 'Percent voting for Clinton', 
    z = 'Median Income'
).update_traces({'marker':{'size': 3}}
               ).update_layout(width=800, height=600)


In [ ]:
def demographics_mult_rmse(a, b, c):
    x = demographics.column('College%')
    z = demographics.column('Percent voting for Clinton')
    y = demographics.column('Median Income')
    
    estimate = a*x + b*z + c
    
    return np.mean((y - estimate)**2)**0.5

In [ ]:
minimize(demographics_mult_rmse)

In [ ]:
coefficients = minimize(demographics_mult_rmse)
coefficients

In [ ]:
50*coefficients.item(0) + 50*coefficients.item(1) + coefficients.item(2)

In [ ]:
coefficients = minimize(demographics_mult_rmse)

a = coefficients.item(0)  # College% coefficient
b = coefficients.item(1)  # Clinton% coefficient
c = coefficients.item(2)  # intercept

fig = px.scatter_3d(
    demographics.to_df(),
    x='College%',
    y='Percent voting for Clinton',
    z='Median Income'
).update_traces({'marker':{'size': 3}}
               ).update_layout(width=800, height=600)

college_grid = np.linspace(
    demographics.column('College%').min(),
    demographics.column('College%').max(),
    30
)

clinton_grid = np.linspace(
    demographics.column('Percent voting for Clinton').min(),
    demographics.column('Percent voting for Clinton').max(),
    30
)

college_mesh, clinton_mesh = np.meshgrid(
    college_grid,
    clinton_grid
)

income_mesh = (
    a * college_mesh
    + b * clinton_mesh
    + c
)

fig.add_surface(
    x=college_mesh,
    y=clinton_mesh,
    z=income_mesh,
    opacity=0.5,
    colorscale='Viridis',
    name='Best-fit plane',
    showscale=False
)

fig.update_layout(
    width=800,
    height=600,
    scene=dict(
        xaxis_title='College%',
        yaxis_title='Percent voting for Clinton',
        zaxis_title='Median Income'
    )
)

In [ ]:
pred_0 = 50*coefficients.item(0) + 0*coefficients.item(1) + coefficients.item(2)
pred_100 = 50*coefficients.item(0) + 100*coefficients.item(1) + coefficients.item(2)

pred_0, pred_100

## HighPeaks

In [ ]:
HighPeaks = Table.read_table('/home/jovyan/shared/Data/HighPeaks.csv')
HighPeaks

In [ ]:
for i in make_array('Elevation', 'Difficulty', 'Ascent', 'Length'):
    HighPeaks.scatter(i, 'Time', fit_line = True)
    r = round(correlation(HighPeaks, i, 'Time'), 3)
    plots.title('correlation: ' + str(r))

In [ ]:
def HighPeaks_mult_rmse(a, b, c, d):
    w = HighPeaks.column('Difficulty')
    x = HighPeaks.column('Ascent')
    z = HighPeaks.column('Length')
    y = HighPeaks.column('Time')
    
    estimate = a*w + b*x + c*z +d
    
    return np.mean((y - estimate)**2)**0.5

In [ ]:
minimize(HighPeaks_mult_rmse)

In [ ]:
HighPeaks_coef = minimize(HighPeaks_mult_rmse)

MtMarcy = HighPeaks.row(0)
MtMarcy

In [ ]:
HighPeaks_coef.item(0) * MtMarcy.item(2) + HighPeaks_coef.item(1) * MtMarcy.item(3) + \
HighPeaks_coef.item(2) * MtMarcy.item(4) + HighPeaks_coef.item(3) 

In [ ]:
slope(HighPeaks, 'Ascent', 'Time') * MtMarcy.item(3) + intercept(HighPeaks, 'Ascent', 'Time')

In [ ]:
HighPeaks_mult_rmse(
    HighPeaks_coef.item(0), 
    HighPeaks_coef.item(1), 
    HighPeaks_coef.item(2), 
    HighPeaks_coef.item(3)
)

In [ ]:
def HighPeaks_Ascent_rmse(a, b):
    x = HighPeaks.column('Ascent')
    y = HighPeaks.column('Time')
    
    estimate = a*x + b
    
    return np.mean((y - estimate)**2)**0.5

In [ ]:
HighPeaks_Ascent_rmse(slope(HighPeaks, 'Ascent', 'Time'), intercept(HighPeaks, 'Ascent', 'Time'))

## Crime Rates

In [ ]:
crime = Table.read_table('/home/jovyan/shared/Data/crime_rates.csv')
crime

In [ ]:
for i in crime.drop(0, 4).labels:
    crime.scatter(i, 'Murder Rate',fit_line = True, s=3)
    r = round(correlation(crime, i, 'Murder Rate'), 3)
    plots.title('correlation: ' + str(r))